In [4]:
from newsapi import NewsApiClient


ImportError: cannot import name 'NewsApiClient' from 'newsapi' (e:\Github Projects 2025\1.Github\quant-trading-system (Not done)\.venv\lib\site-packages\newsapi\__init__.py)

In [ ]:
top_headlines = newsapi.get_top_headlines(q='bitcoin',
                                          sources='bbc-news,the-verge',
                                          category='business',
                                          language='en',
                                          country='us')

In [7]:
import requests, pandas as pd, re
from datetime import datetime, timedelta

In [12]:
API_KEY = "2415540e75e74163a76cfbf93ded9e1a"
QUERY = "FDA OR earnings OR acquisition OR upgrade OR contract"
LIMIT = 50
date_str = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
def get_general_news():
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": QUERY,
        "from": date_str, # datetime.now().strftime('%Y-%m-%d'),
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": LIMIT,
        "apiKey": API_KEY
    }
    response = requests.get(url, params=params)
    # print("Status:", response.status_code)
    # print("Response:", response.json())
    if response.status_code != 200:
        print("Error fetching news:", response.status_code, response.text)
        return []
    else:
        return response.json().get("articles", [])

def extract_tickers(text):
    return re.findall(r'\b[A-Z]{2,5}\b', text)

def build_catalyst_news_list():
    articles = get_general_news()
    rows = []
    for a in articles:
        title, desc = a.get("title",""), a.get("description","")
        tickers = set(extract_tickers(title + " " + desc))
        rows.append({
            "publishedAt": a.get("publishedAt"),
            "source": a.get("source",{}).get("name"),
            "title": title,
            "description": desc,
            "url": a.get("url"),
            "tickers": ", ".join(tickers)
        })
    df = pd.DataFrame(rows)
    out = f"catalyst_news_{datetime.now().strftime('%Y-%m-%d')}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)

In [13]:
get_general_news()

[{'source': {'id': None, 'name': 'Upenn.edu'},
  'author': 'Gloria Yuen',
  'title': 'How Are Freelancers Adapting to Gen AI?',
  'description': 'Wharton research reveals how freelancers are adapting to stay competitive after the launch of ChatGPT.…Read\xa0More',
  'url': 'https://knowledge.wharton.upenn.edu/article/how-are-freelancers-adapting-to-gen-ai/',
  'urlToImage': 'https://knowledge.wharton.upenn.edu/wp-content/uploads/2025/06/6.11.25-raj-freelancers-ai-GettyImages-1765762402-600x500.jpg',
  'publishedAt': '2025-06-10T16:25:55Z',
  'content': 'Freelancers have long been hailed as the agile foot soldiers of the digital economy. But when ChatGPT launched out of nowhere in late 2022, many of these independent workers found themselves adapting… [+5611 chars]'},
 {'source': {'id': None, 'name': 'Bringatrailer.com'},
  'author': 'bringatrailer',
  'title': '1970 Toyota Land Cruiser FJ40 Project at No Reserve',
  'description': "This 1970 Toyota Land Cruiser FJ40 is said to have been

In [16]:
build_catalyst_news_list()

Saved: catalyst_news_2025-06-11.csv


In [20]:
import requests
import json
import pandas as pd

# URLs for exchange symbol lists
url_nasdaq = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.json"
url_nyse   = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.json"
# url_amex   = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/amex/amex_tickers.json"

def load_symbols(url):
    return json.loads(requests.get(url).text)

# Combine ticker symbols
tickers = set(load_symbols(url_nasdaq) + load_symbols(url_nyse)) # + load_symbols(url_amex))

In [22]:
df = pd.DataFrame({"symbol": sorted(tickers)})
# Save to CSV
df = pd.DataFrame({"symbol": sorted(tickers)})
df.to_csv("../data/tickers.csv", index=False)